# return_preserving_hybrid_target_bc_replay_v1

Purpose: screen return-preserving hybrid state-objective targets after `wf2020_riskcap_full_bc_replay_v1` proved the full risk-cap replacement is risk-safe but underparticipates.

Governance:
- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `wf2020_riskcap_full_bc_replay_v1`; reference baseline `soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05`.
- Role: diagnose. This card is not allowed to promote or retain an actor.
- Target window: required walk-forward windows `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Accepted trade-off: keep MaxDD <20% while recovering enough return/relative return to close the `walkforward_relative_return_gap`.
- Changed layer: `model_selection`, `training_target`, `bc_warm_start`, `sample_weighting`, `official_replay`.
- External reference: local SOXLBacktest qlib buy/sell risk-cap reference and previous model-retention frontier.
- Hypothesis: restoring non-risk target exposure while preserving the risk-cap floor, plus modest class-balanced BC weighting, can restore participation without losing the drawdown gate.
- Acceptance criteria: one global candidate passes all three fold gates; otherwise no training/retention is allowed and the next action is a narrower target/weight repair.
- Expected failure mode: class balancing or restored exposure overtrades 2022/2023-2024, or still underparticipates because the actor predicts dominant low-exposure actions.
- Closest historical success: `bull_reentry_three_fold_checkpoint_metric_replay_v1` passed `wf_2022` and `wf_2023_2024` but failed `wf_2020` return floor.
- Closest historical failure: `wf2020_riskcap_full_bc_replay_v1` passed 3/3 MaxDD gates but failed return/relative gates through underparticipation.
- Why not duplicate: this card screens new hybrid target contracts and BC weight settings, not the already-executed full risk-cap target.


In [1]:

from __future__ import annotations

import json
import math
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics, max_drawdown_interval_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import build_warm_start_replay_segment, slice_warm_start_daily_path
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'return_preserving_hybrid_target_bc_replay_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / 'runtime/model_selection/card/wf2020_risk_cap_reference_target_screen_v1'
DIRECT_REPLAY_CARD = ROOT / 'runtime/model_selection/card/wf2020_riskcap_full_bc_replay_v1'
GATE_CONTRACT_PATH = ROOT / 'runtime/model_selection/card/wf2020_reference_floor_gate_calibration_v1/training_gate_contract.json'
PREVIOUS_REPLAY_PATH = ROOT / 'runtime/model_selection/card/bull_reentry_three_fold_checkpoint_metric_replay_v1/checkpoint_selection_by_fold.csv'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_MAX_DD = 20.0

BASE_CONTRACT_PATH = SOURCE_CARD / 'current_bull_reentry_contract_combined_target_contract.csv'
FULL_RISKCAP_CONTRACT_PATH = SOURCE_CARD / 'riskcap_full_replacement_combined_target_contract.csv'

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2019-12-31'),
        'valid': ('2020-01-01', '2020-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=240, lr=0.001, batch_size=128, seed=SEED)
WEIGHT_CONFIGS = {
    'riskcap_default_weight': StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=4.0,
        downtrend_multiplier=2.0,
        release_gate_multiplier=10.0,
        ordinary_multiplier=1.0,
        objective_weight_power=0.0,
        class_balanced_beta=0.0,
        weight_floor=0.05,
        weight_cap=30.0,
        normalize_mean=True,
    ),
    'participation_balanced_weight': StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=5.0,
        downtrend_multiplier=2.0,
        release_gate_multiplier=10.0,
        ordinary_multiplier=1.5,
        objective_weight_power=0.25,
        class_balanced_beta=0.40,
        weight_floor=0.05,
        weight_cap=40.0,
        normalize_mean=True,
    ),
}
SLICE_COLS = (
    'risk_cap_reference_active_state',
    'risk_cap_reference_buy_state',
    'risk_cap_reference_sell_state',
    'release_gate_state',
    'is_downtrend_state',
    'drawdown_control_state',
    'hybrid_restore_state',
    'hybrid_blend_state',
)


def json_default(value: Any) -> Any:
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(
        json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n',
        encoding='utf-8',
    )


def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def datetime_index_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if isinstance(out.index, pd.MultiIndex):
        if 'instrument' in out.index.names:
            out = out.droplevel('instrument')
        else:
            out = out.reset_index(level=list(range(1, out.index.nlevels)), drop=True)
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index))
    return out.sort_index()


def datetime_index_series(series: pd.Series) -> pd.Series:
    out = series.copy()
    if isinstance(out.index, pd.MultiIndex):
        if 'instrument' in out.index.names:
            out = out.droplevel('instrument')
        else:
            out.index = out.index.get_level_values(0)
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index))
    return out.sort_index()


def truthy(series: pd.Series) -> pd.Series:
    return series.fillna(False).astype(bool)


def selected_target_position(frame: pd.DataFrame) -> pd.Series:
    return pd.Series(
        [float(state_objective_target_position_from_row(row)) for row in frame.to_dict('records')],
        index=frame.index,
        dtype=float,
    ).clip(0.0, 1.0)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:

profile = load_profile(PROFILE_NAME)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
source_summary = json.loads((SOURCE_CARD / 'summary.json').read_text(encoding='utf-8'))
direct_replay_summary = json.loads((DIRECT_REPLAY_CARD / 'summary.json').read_text(encoding='utf-8'))
gate_contract = json.loads(GATE_CONTRACT_PATH.read_text(encoding='utf-8'))
base_targets = pd.read_csv(BASE_CONTRACT_PATH, parse_dates=['datetime'])
full_targets = pd.read_csv(FULL_RISKCAP_CONTRACT_PATH, parse_dates=['datetime'])
previous_replay = pd.read_csv(PREVIOUS_REPLAY_PATH) if PREVIOUS_REPLAY_PATH.exists() else pd.DataFrame()

if len(base_targets) != len(full_targets):
    raise ValueError('base/full target contracts must have equal row counts')
if not base_targets['datetime'].equals(full_targets['datetime']):
    raise ValueError('base/full target contracts are not date-aligned')


def make_hybrid_contract(name: str) -> pd.DataFrame:
    out = full_targets.copy()
    for column in ('hybrid_restore_state', 'hybrid_blend_state'):
        out[column] = False
    wf_mask = out['fold'].astype(str).eq('wf_2020_train')
    base_wf = base_targets.loc[wf_mask].reset_index(drop=True)
    hybrid_wf = out.loc[wf_mask].reset_index(drop=True)
    base_position = selected_target_position(base_wf)
    full_position = selected_target_position(hybrid_wf)
    active = truthy(hybrid_wf['risk_cap_reference_active_state'])
    downtrend = truthy(hybrid_wf['is_downtrend_state'])
    drawdown = truthy(hybrid_wf['drawdown_control_state'])
    nonrisk = (~active) & (~downtrend) & (~drawdown)

    if name == 'hybrid_restore_nonrisk_cap50':
        mask = nonrisk
        target = np.minimum(base_position, 0.50)
        state_column = 'hybrid_restore_state'
    elif name == 'hybrid_restore_nonrisk_cap65':
        mask = nonrisk
        target = np.minimum(base_position, 0.65)
        state_column = 'hybrid_restore_state'
    elif name == 'hybrid_blend_nonrisk_50':
        mask = nonrisk
        target = np.minimum(0.5 * base_position + 0.5 * full_position, 0.50)
        state_column = 'hybrid_blend_state'
    else:
        raise ValueError(f'unknown hybrid contract: {name}')

    loc = out.index[wf_mask]
    mask_np = mask.to_numpy(dtype=bool)
    target_np = pd.Series(target, index=hybrid_wf.index).to_numpy(dtype=float)
    restore_cols = ('teacher_position', 'rebound_target_position', 'risk_target_position', 'risk_target_position_original')
    for column in restore_cols:
        values = pd.to_numeric(hybrid_wf[column], errors='coerce').fillna(0.0).clip(0.0, 1.0).to_numpy(dtype=float)
        values[mask_np] = target_np[mask_np]
        out.loc[loc, column] = values

    objective = pd.to_numeric(hybrid_wf['objective_weight'], errors='coerce').fillna(1.0).to_numpy(dtype=float)
    objective[mask_np] = np.maximum(objective[mask_np], 6.0)
    out.loc[loc, 'objective_weight'] = objective
    out.loc[loc[mask_np], state_column] = True
    out.loc[loc[mask_np], 'source_components'] = out.loc[loc[mask_np], 'source_components'].astype(str) + ',return_preserving_hybrid_restore'
    out.loc[loc[mask_np], 'source_contract_id'] = EXPERIMENT
    out.loc[loc[mask_np], 'target_semantics_revision'] = name
    out.loc[:, 'target_variant'] = name
    return out

candidate_variants = [
    'hybrid_restore_nonrisk_cap50',
    'hybrid_restore_nonrisk_cap65',
    'hybrid_blend_nonrisk_50',
]
contracts = {name: make_hybrid_contract(name) for name in candidate_variants}

manifest_rows = []
distribution_rows = []
for name, contract in contracts.items():
    contract_path = CARD / f'{name}_combined_target_contract.csv'
    contract.to_csv(contract_path, index=False)
    manifest_rows.append({'variant': name, 'path': str(contract_path.relative_to(ROOT))})
    for fold_name in ('wf_2020_train', 'wf_2022_train', 'wf_2023_2024_train'):
        fold_frame = contract[contract['fold'].astype(str).eq(fold_name)].copy()
        positions = selected_target_position(fold_frame)
        distribution_rows.append({
            'variant': name,
            'fold': fold_name,
            'rows': int(len(fold_frame)),
            'avg_target_position': float(positions.mean()) if len(positions) else np.nan,
            'median_target_position': float(positions.median()) if len(positions) else np.nan,
            'target_ge_15_rows': int((positions >= 0.15).sum()),
            'target_ge_35_rows': int((positions >= 0.35).sum()),
            'target_ge_50_rows': int((positions >= 0.50).sum()),
            'target_ge_65_rows': int((positions >= 0.65).sum()),
            'hybrid_restore_rows': int(fold_frame.get('hybrid_restore_state', pd.Series(False, index=fold_frame.index)).fillna(False).astype(bool).sum()),
            'hybrid_blend_rows': int(fold_frame.get('hybrid_blend_state', pd.Series(False, index=fold_frame.index)).fillna(False).astype(bool).sum()),
            'riskcap_active_rows': int(fold_frame.get('risk_cap_reference_active_state', pd.Series(False, index=fold_frame.index)).fillna(False).astype(bool).sum()),
        })

candidate_contract_manifest = pd.DataFrame(manifest_rows)
candidate_target_distribution = pd.DataFrame(distribution_rows)
weight_config_manifest = pd.DataFrame([
    {'weight_config': name, **asdict(config)}
    for name, config in WEIGHT_CONFIGS.items()
])
candidate_contract_manifest.to_csv(CARD / 'candidate_contract_manifest.csv', index=False)
candidate_target_distribution.to_csv(CARD / 'candidate_target_distribution.csv', index=False)
weight_config_manifest.to_csv(CARD / 'candidate_weight_config_manifest.csv', index=False)

init_qlib(QLIB_DATA_PATH)

display(candidate_target_distribution)
display(weight_config_manifest)


[11345:MainThread](2026-06-14 09:34:49,778) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[11345:MainThread](2026-06-14 09:34:50,292) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[11345:MainThread](2026-06-14 09:34:50,294) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


,variant,fold,rows,avg_target_position,median_target_position,target_ge_15_rows,target_ge_35_rows,target_ge_50_rows,target_ge_65_rows,hybrid_restore_rows,hybrid_blend_rows,riskcap_active_rows
0,hybrid_restore_nonrisk_cap50,wf_2020_train,2470,0.255472,0.2007,1395,894,584,0,1586,0,424
1,hybrid_restore_nonrisk_cap50,wf_2022_train,505,0.348817,0.2500,478,189,114,71,0,0,0
2,hybrid_restore_nonrisk_cap50,wf_2023_2024_train,756,0.277530,0.2000,631,189,114,71,0,0,0
3,hybrid_restore_nonrisk_cap65,wf_2020_train,2470,0.265538,0.2007,1395,894,584,84,1586,0,424
4,hybrid_restore_nonrisk_cap65,wf_2022_train,505,0.348817,0.2500,478,189,114,71,0,0,0
5,hybrid_restore_nonrisk_cap65,wf_2023_2024_train,756,0.277530,0.2000,631,189,114,71,0,0,0
6,hybrid_blend_nonrisk_50,wf_2020_train,2470,0.164759,0.1375,1168,133,0,0,0,1586,424
7,hybrid_blend_nonrisk_50,wf_2022_train,505,0.348817,0.2500,478,189,114,71,0,0,0
8,hybrid_blend_nonrisk_50,wf_2023_2024_train,756,0.277530,0.2000,631,189,114,71,0,0,0


,weight_config,rebound_multiplier,downtrend_multiplier,release_gate_multiplier,ordinary_multiplier,objective_weight_power,class_balanced_beta,weight_floor,weight_cap,normalize_mean
0,riskcap_default_weight,4.0,2.0,10.0,1.0,0.00,0.0,0.05,30.0,True
1,participation_balanced_weight,5.0,2.0,10.0,1.5,0.25,0.4,0.05,40.0,True


In [3]:

coverage_rows: list[dict[str, Any]] = []
fold_cache: dict[str, dict[str, Any]] = {}
for fold, spec in WALK_FORWARD_WINDOWS.items():
    train_start, train_end = map(pd.Timestamp, spec['train'])
    test_start, test_end = map(pd.Timestamp, spec['test'])
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = datetime_index_frame(flatten_feature_frame(dataset.handler.feature_frame))
    price_full = datetime_index_series(dataset.handler.price_frame['price'])
    train_frame = feature_full.loc[spec['train'][0]:spec['train'][1]].copy()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec['test'], lookback=LOOKBACK)
    fold_cache[fold] = {
        'dataset': dataset,
        'feature_full': feature_full,
        'price_full': price_full,
        'train_frame': train_frame,
        'warm_seg': warm_seg,
    }
    for variant, contract in contracts.items():
        target_fold = str(spec['target_fold'])
        fold_targets = contract[contract['fold'].astype(str).eq(target_fold)].copy()
        train_targets = fold_targets[(fold_targets['datetime'] >= train_start) & (fold_targets['datetime'] <= train_end)].copy()
        overlap = train_targets[(train_targets['datetime'] >= test_start) & (train_targets['datetime'] <= test_end)]
        coverage_rows.append({
            'variant': variant,
            'fold': fold,
            'target_fold': target_fold,
            'target_rows': int(len(train_targets)),
            'target_start': str(train_targets['datetime'].min().date()) if len(train_targets) else '',
            'target_end': str(train_targets['datetime'].max().date()) if len(train_targets) else '',
            'train_start': spec['train'][0],
            'train_end': spec['train'][1],
            'test_start': spec['test'][0],
            'test_end': spec['test'][1],
            'feature_start': str(feature_full.index.min().date()),
            'feature_end': str(feature_full.index.max().date()),
            'train_frame_rows': int(len(train_frame)),
            'warm_start': warm_seg[0],
            'warm_end': warm_seg[1],
            'eval_overlap_rows': int(len(overlap)),
            'fold_local_gate': bool(len(train_targets) > 0 and len(overlap) == 0),
        })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
coverage


[11345:MainThread](2026-06-14 09:34:51,217) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[11345:MainThread](2026-06-14 09:34:51,217) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[11345:MainThread](2026-06-14 09:34:51,218) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[11345:MainThread](2026-06-14 09:34:52,308) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[11345:MainThread](2026-06-14 09:34:52,308) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[11345:MainThread](2026-06-14 09:34:52,309) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[11345:MainThread](2026-06-14 09:34:53,583) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[11345:MainThread](2026-06-14 09:34:53,583) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[11345:MainThread](2026-06-14 09:34:53,584) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


,variant,fold,target_fold,target_rows,target_start,target_end,train_start,train_end,test_start,test_end,feature_start,feature_end,train_frame_rows,warm_start,warm_end,eval_overlap_rows,fold_local_gate
0,hybrid_restore_nonrisk_cap50,wf_2020,wf_2020_train,2470,2010-03-11,2019-12-31,2010-03-11,2019-12-31,2020-01-01,2020-12-31,2011-03-11,2020-12-31,2217,2019-10-07,2020-12-31,0,True
1,hybrid_restore_nonrisk_cap65,wf_2020,wf_2020_train,2470,2010-03-11,2019-12-31,2010-03-11,2019-12-31,2020-01-01,2020-12-31,2011-03-11,2020-12-31,2217,2019-10-07,2020-12-31,0,True
2,hybrid_blend_nonrisk_50,wf_2020,wf_2020_train,2470,2010-03-11,2019-12-31,2010-03-11,2019-12-31,2020-01-01,2020-12-31,2011-03-11,2020-12-31,2217,2019-10-07,2020-12-31,0,True
3,hybrid_restore_nonrisk_cap50,wf_2022,wf_2022_train,253,2020-01-02,2020-12-31,2010-03-11,2020-12-31,2022-01-01,2022-12-31,2011-03-11,2022-12-30,2470,2021-10-07,2022-12-30,0,True
4,hybrid_restore_nonrisk_cap65,wf_2022,wf_2022_train,253,2020-01-02,2020-12-31,2010-03-11,2020-12-31,2022-01-01,2022-12-31,2011-03-11,2022-12-30,2470,2021-10-07,2022-12-30,0,True
5,hybrid_blend_nonrisk_50,wf_2022,wf_2022_train,253,2020-01-02,2020-12-31,2010-03-11,2020-12-31,2022-01-01,2022-12-31,2011-03-11,2022-12-30,2470,2021-10-07,2022-12-30,0,True
6,hybrid_restore_nonrisk_cap50,wf_2023_2024,wf_2023_2024_train,756,2020-01-02,2022-12-30,2010-03-11,2022-12-31,2024-01-01,2024-12-31,2011-03-11,2024-12-31,2973,2023-10-05,2024-12-31,0,True
7,hybrid_restore_nonrisk_cap65,wf_2023_2024,wf_2023_2024_train,756,2020-01-02,2022-12-30,2010-03-11,2022-12-31,2024-01-01,2024-12-31,2011-03-11,2024-12-31,2973,2023-10-05,2024-12-31,0,True
8,hybrid_blend_nonrisk_50,wf_2023_2024,wf_2023_2024_train,756,2020-01-02,2022-12-30,2010-03-11,2022-12-31,2024-01-01,2024-12-31,2011-03-11,2024-12-31,2973,2023-10-05,2024-12-31,0,True


In [4]:

metric_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []
drawdown_rows: list[dict[str, Any]] = []
daily_manifest: list[dict[str, Any]] = []

relative_targets = gate_contract.get('target_relative_return_by_fold_pp', {})
previous_by_fold = previous_replay.set_index('fold').to_dict('index') if not previous_replay.empty and 'fold' in previous_replay else {}

for variant_index, variant in enumerate(candidate_variants):
    contract = contracts[variant]
    for weight_index, (weight_name, weight_config) in enumerate(WEIGHT_CONFIGS.items()):
        candidate = f'{variant}__{weight_name}'
        for fold_index, (fold, spec) in enumerate(WALK_FORWARD_WINDOWS.items()):
            print(f"[candidate:{candidate}][fold:{fold}] train={spec['train']} test={spec['test']} target_fold={spec['target_fold']}")
            cache = fold_cache[fold]
            target_fold = str(spec['target_fold'])
            train_start, train_end = map(pd.Timestamp, spec['train'])
            fold_targets = contract[contract['fold'].astype(str).eq(target_fold)].copy()
            fold_targets = fold_targets[(fold_targets['datetime'] >= train_start) & (fold_targets['datetime'] <= train_end)].copy()
            if fold_targets.empty:
                metric_rows.append({'candidate': candidate, 'variant': variant, 'weight_config': weight_name, 'fold': fold, 'status': 'skipped_no_targets'})
                continue
            seed = SEED + variant_index * 101 + weight_index * 37 + fold_index * 17
            set_seed(seed)
            fit = fit_state_objective_bc_actor_from_frame(
                cache['train_frame'],
                targets=fold_targets,
                profile=profile,
                train_config=BC_TRAIN_CONFIG,
                sample_weight_config=weight_config,
                sample_size=0,
                rng_seed=seed,
            )
            fit_rows.append({
                'candidate': candidate,
                'variant': variant,
                'weight_config': weight_name,
                'fold': fold,
                'target_fold': spec['target_fold'],
                'target_rows': int(len(fold_targets)),
                'bc_samples': int(len(fit.actions)),
                'unique_actions': int(pd.Series(fit.actions).nunique()) if len(fit.actions) else 0,
                **{k: float(v) if isinstance(v, (int, float, np.number)) and np.isfinite(float(v)) else v for k, v in fit.metrics.items()},
            })
            bc_dataset = build_state_objective_bc_dataset_from_frame(
                cache['train_frame'],
                targets=fold_targets,
                profile=profile,
                sample_size=0,
                rng_seed=seed,
            )
            slice_metrics = evaluate_bc_actor_slice_fit(
                fit.actor,
                fit.obs,
                fit.actions,
                bc_dataset.metadata,
                slice_cols=SLICE_COLS,
            )
            slice_metrics.insert(0, 'fold', fold)
            slice_metrics.insert(0, 'weight_config', weight_name)
            slice_metrics.insert(0, 'variant', variant)
            slice_metrics.insert(0, 'candidate', candidate)
            slice_frames.append(slice_metrics)
            action_dist = fit.action_distribution.copy()
            action_dist.insert(0, 'fold', fold)
            action_dist.insert(0, 'weight_config', weight_name)
            action_dist.insert(0, 'variant', variant)
            action_dist.insert(0, 'candidate', candidate)
            action_frames.append(action_dist)

            bt = backtest_profile(
                fit.actor,
                cache['feature_full'],
                cache['price_full'],
                position_cap_config=profile.position_cap,
                position_budget_config=profile.position_budget,
                execution_config=profile.execution,
                mode_router_config=profile.mode_router,
                asset_idx=0,
                feature_columns=feature_columns,
                model_config=profile.model,
                test_seg=cache['warm_seg'],
            )
            daily = slice_warm_start_daily_path(
                portfolio_df=bt['portfolio'],
                price_series=cache['price_full'],
                eval_seg=spec['test'],
                position_history=bt.get('position_history', []),
                trade_signals=bt.get('trade_signals', []),
            )
            daily_path = CARD / f'{candidate}_{fold}_official_daily_path.csv'
            daily.to_csv(daily_path, index=False)
            metrics = daily_path_metrics(daily)
            drawdown = max_drawdown_interval_metrics(daily)
            drawdown_rows.append({'candidate': candidate, 'variant': variant, 'weight_config': weight_name, 'fold': fold, **drawdown, 'daily_path': str(daily_path.relative_to(ROOT))})
            daily_manifest.append({
                'candidate': candidate,
                'variant': variant,
                'weight_config': weight_name,
                'fold': fold,
                'path': str(daily_path.relative_to(ROOT)),
                'rows': int(len(daily)),
                'start': metrics.get('start'),
                'end': metrics.get('end'),
            })

            previous = previous_by_fold.get(fold, {})
            target_relative = float(relative_targets.get(fold, 30.0))
            return_floor = float(gate_contract.get('wf2020_reference_floor_return_pct', math.nan)) if fold == 'wf_2020' else float('-inf')
            max_dd_gate = bool(float(metrics['max_dd']) < TARGET_MAX_DD)
            relative_gate = bool(float(metrics['relative_return']) >= target_relative)
            return_floor_gate = bool(fold != 'wf_2020' or float(metrics['return']) >= return_floor)
            fold_gate = bool(max_dd_gate and relative_gate and return_floor_gate)
            metric_rows.append({
                'candidate': candidate,
                'variant': variant,
                'weight_config': weight_name,
                'fold': fold,
                'status': 'evaluated',
                'target_fold': spec['target_fold'],
                'warm_start': cache['warm_seg'][0],
                'warm_end': cache['warm_seg'][1],
                'target_rows': int(len(fold_targets)),
                'bc_samples': int(len(fit.actions)),
                'return': float(metrics['return']),
                'buy_and_hold_return': float(metrics['buy_and_hold_return']),
                'relative_return': float(metrics['relative_return']),
                'max_dd': float(metrics['max_dd']),
                'avg_pos': float(metrics['avg_pos']),
                'trade_count': int(pd.Series(daily['action']).astype(str).str.upper().isin(['BUY', 'SELL']).sum()),
                'target_relative_return': target_relative,
                'target_max_dd': TARGET_MAX_DD,
                'wf2020_reference_floor_return': return_floor if math.isfinite(return_floor) else np.nan,
                'relative_gate': relative_gate,
                'max_dd_gate': max_dd_gate,
                'return_floor_gate': return_floor_gate,
                'fold_gate': fold_gate,
                'previous_selected_relative_return': previous.get('selected_relative_return', np.nan),
                'previous_selected_max_dd': previous.get('selected_max_dd', np.nan),
                'relative_delta_vs_previous': float(metrics['relative_return']) - float(previous.get('selected_relative_return', np.nan)) if previous else np.nan,
                'max_dd_delta_vs_previous': float(metrics['max_dd']) - float(previous.get('selected_max_dd', np.nan)) if previous else np.nan,
                'daily_path': str(daily_path.relative_to(ROOT)),
            })

fit_metrics = pd.DataFrame(fit_rows)
fold_metrics = pd.DataFrame(metric_rows)
slice_metrics = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()
drawdown_intervals = pd.DataFrame(drawdown_rows)
daily_manifest_df = pd.DataFrame(daily_manifest)

fit_metrics.to_csv(CARD / 'bc_fit_metrics.csv', index=False)
fold_metrics.to_csv(CARD / 'fold_candidate_replay_metrics.csv', index=False)
slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
action_distribution.to_csv(CARD / 'bc_action_distribution.csv', index=False)
drawdown_intervals.to_csv(CARD / 'drawdown_interval_diagnostic.csv', index=False)
daily_manifest_df.to_csv(CARD / 'daily_path_manifest.csv', index=False)

fold_metrics


[candidate:hybrid_restore_nonrisk_cap50__riskcap_default_weight][fold:wf_2020] train=('2010-03-11', '2019-12-31') test=('2020-01-01', '2020-12-31') target_fold=wf_2020_train


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[11345:MainThread](2026-06-14 09:34:59,263) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!


[11345:MainThread](2026-06-14 09:34:59,263) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


[11345:MainThread](2026-06-14 09:34:59,266) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x1216ba5d0>


...

backtest loop:   0%|          | 0/311 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,candidate,variant,weight_config,fold,status,target_fold,warm_start,warm_end,target_rows,bc_samples,...,wf2020_reference_floor_return,relative_gate,max_dd_gate,return_floor_gate,fold_gate,previous_selected_relative_return,previous_selected_max_dd,relative_delta_vs_previous,max_dd_delta_vs_previous,daily_path
0,hybrid_restore_nonrisk_cap50__riskcap_default_...,hybrid_restore_nonrisk_cap50,riskcap_default_weight,wf_2020,evaluated,wf_2020_train,2019-10-07,2020-12-31,2470,2156,...,69.604552,True,False,False,False,-277.735388,15.489488,212.433813,18.745407,runtime/model_selection/card/return_preserving...
1,hybrid_restore_nonrisk_cap50__riskcap_default_...,hybrid_restore_nonrisk_cap50,riskcap_default_weight,wf_2022,evaluated,wf_2022_train,2021-10-07,2022-12-30,253,252,...,NaN,True,False,True,False,65.864820,18.802284,-0.326203,2.942193,runtime/model_selection/card/return_preserving...
2,hybrid_restore_nonrisk_cap50__riskcap_default_...,hybrid_restore_nonrisk_cap50,riskcap_default_weight,wf_2023_2024,evaluated,wf_2023_2024_train,2023-10-05,2024-12-31,756,755,...,NaN,False,True,True,False,35.566229,8.955769,-21.184206,5.695730,runtime/model_selection/card/return_preserving...
3,hybrid_restore_nonrisk_cap50__participation_ba...,hybrid_restore_nonrisk_cap50,participation_balanced_weight,wf_2020,evaluated,wf_2020_train,2019-10-07,2020-12-31,2470,2156,...,69.604552,True,False,False,False,-277.735388,15.489488,219.490213,19.074601,runtime/model_selection/card/return_preserving...
4,hybrid_restore_nonrisk_cap50__participation_ba...,hybrid_restore_nonrisk_cap50,participation_balanced_weight,wf_2022,evaluated,wf_2022_train,2021-10-07,2022-12-30,253,252,...,NaN,True,False,True,False,65.864820,18.802284,-9.411003,11.933391,runtime/model_selection/card/return_preserving...
5,hybrid_restore_nonrisk_cap50__participation_ba...,hybrid_restore_nonrisk_cap50,participation_balanced_weight,wf_2023_2024,evaluated,wf_2023_2024_train,2023-10-05,2024-12-31,756,755,...,NaN,False,True,True,False,35.566229,8.955769,-18.202744,4.010237,runtime/model_selection/card/return_preserving...
6,hybrid_restore_nonrisk_cap65__riskcap_default_...,hybrid_restore_nonrisk_cap65,riskcap_default_weight,wf_2020,evaluated,wf_2020_train,2019-10-07,2020-12-31,2470,2156,...,69.604552,True,False,False,False,-277.735388,15.489488,232.871542,19.456420,runtime/model_selection/card/return_preserving...
7,hybrid_restore_nonrisk_cap65__riskcap_default_...,hybrid_restore_nonrisk_cap65,riskcap_default_weight,wf_2022,evaluated,wf_2022_train,2021-10-07,2022-12-30,253,252,...,NaN,True,False,True,False,65.864820,18.802284,-0.326203,2.942193,runtime/model_selection/card/return_preserving...
8,hybrid_restore_nonrisk_cap65__riskcap_default_...,hybrid_restore_nonrisk_cap65,riskcap_default_weight,wf_2023_2024,evaluated,wf_2023_2024_train,2023-10-05,2024-12-31,756,755,...,NaN,False,True,True,False,35.566229,8.955769,-21.184206,5.695730,runtime/model_selection/card/return_preserving...
9,hybrid_restore_nonrisk_cap65__participation_ba...,hybrid_restore_nonrisk_cap65,participation_balanced_weight,wf_2020,evaluated,wf_2020_train,2019-10-07,2020-12-31,2470,2156,...,69.604552,True,False,False,False,-277.735388,15.489488,225.282873,19.456420,runtime/model_selection/card/return_preserving...


In [5]:

evaluated = fold_metrics[fold_metrics['status'].eq('evaluated')].copy()
required_folds = list(WALK_FORWARD_WINDOWS)

if evaluated.empty:
    candidate_score = pd.DataFrame()
    selected_candidate = ''
    selected_rows = evaluated.copy()
else:
    candidate_rows = []
    for candidate, group in evaluated.groupby('candidate'):
        present = set(group['fold'].astype(str))
        missing = [fold for fold in required_folds if fold not in present]
        rel_gap = []
        wf2020_return_gap = 0.0
        for row in group.to_dict('records'):
            rel_gap.append(max(0.0, float(row['target_relative_return']) - float(row['relative_return'])))
            if row['fold'] == 'wf_2020':
                floor = float(row['wf2020_reference_floor_return']) if pd.notna(row['wf2020_reference_floor_return']) else 0.0
                wf2020_return_gap = max(0.0, floor - float(row['return']))
        worst_max_dd = float(group['max_dd'].max())
        max_dd_gap = max(0.0, worst_max_dd - TARGET_MAX_DD)
        candidate_rows.append({
            'candidate': candidate,
            'variant': group['variant'].iloc[0],
            'weight_config': group['weight_config'].iloc[0],
            'folds_present': ','.join(sorted(present)),
            'missing_folds': ','.join(missing),
            'fold_gate_pass_count': int(group['fold_gate'].astype(bool).sum()),
            'max_dd_pass_count': int(group['max_dd_gate'].astype(bool).sum()),
            'relative_pass_count': int(group['relative_gate'].astype(bool).sum()),
            'return_floor_pass_count': int(group['return_floor_gate'].astype(bool).sum()),
            'worst_max_dd': worst_max_dd,
            'max_dd_gap': float(max_dd_gap),
            'min_relative_return': float(group['relative_return'].min()),
            'max_relative_gap': float(max(rel_gap) if rel_gap else 0.0),
            'wf2020_return_gap': float(wf2020_return_gap),
            'mean_avg_pos': float(group['avg_pos'].mean()),
            'mean_return': float(group['return'].mean()),
        })
    candidate_score = pd.DataFrame(candidate_rows).sort_values(
        [
            'fold_gate_pass_count',
            'max_dd_pass_count',
            'relative_pass_count',
            'return_floor_pass_count',
            'max_dd_gap',
            'max_relative_gap',
            'wf2020_return_gap',
            'worst_max_dd',
        ],
        ascending=[False, False, False, False, True, True, True, True],
    ).reset_index(drop=True)
    selected_candidate = str(candidate_score.iloc[0]['candidate']) if len(candidate_score) else ''
    selected_rows = evaluated[evaluated['candidate'].eq(selected_candidate)].copy()

candidate_score.to_csv(CARD / 'candidate_scorecard.csv', index=False)
selected_rows.to_csv(CARD / 'fold_replay_metrics.csv', index=False)
selected_contract_path = ''
if selected_candidate:
    selected_variant = str(selected_rows['variant'].iloc[0])
    selected_contract = contracts[selected_variant].copy()
    selected_contract_path = str((CARD / 'selected_hybrid_target_contract.csv').relative_to(ROOT))
    selected_contract.to_csv(CARD / 'selected_hybrid_target_contract.csv', index=False)
else:
    selected_variant = ''

missing_folds = [fold for fold in required_folds if fold not in set(selected_rows['fold'].astype(str))]
failed_folds = selected_rows.loc[~selected_rows['fold_gate'].astype(bool), 'fold'].astype(str).tolist() if not selected_rows.empty else required_folds
full_gate = bool((not selected_rows.empty) and not missing_folds and selected_rows['fold_gate'].astype(bool).all())
max_dd_pass_count = int(selected_rows['max_dd_gate'].sum()) if not selected_rows.empty else 0
relative_pass_count = int(selected_rows['relative_gate'].sum()) if not selected_rows.empty else 0
return_floor_pass_count = int(selected_rows['return_floor_gate'].sum()) if not selected_rows.empty else 0

if full_gate:
    decision = 'return_preserving_hybrid_bc_three_fold_gate_pass_no_promotion'
    next_mechanism = 'run_fixed_split_ytd_retention_training_and_actor_review_before_any_promotion'
else:
    decision = 'return_preserving_hybrid_bc_replay_no_promotion'
    next_mechanism = 'inspect_selected_hybrid_failure_and_repair_participation_weight_or_target_before_retention_training'

selected_by_fold = {
    str(row['fold']): {
        'fold': str(row['fold']),
        'status': str(row['status']),
        'selected_candidate': str(row['candidate']),
        'selected_variant': str(row['variant']),
        'selected_weight_config': str(row['weight_config']),
        'selected_kind': 'target_weight_bc_initialization',
        'selected_iter': 0,
        'selected_return': float(row['return']),
        'selected_relative_return': float(row['relative_return']),
        'selected_max_dd': float(row['max_dd']),
        'selected_avg_pos': float(row['avg_pos']),
        'selected_target_gate': bool(row['fold_gate']),
        'selected_candidate_gate': bool(row['fold_gate']),
        'selected_drift_gate': bool(row['max_dd_gate']),
        'target_relative_return': float(row['target_relative_return']),
        'target_max_dd': float(row['target_max_dd']),
        'daily_path': row['daily_path'],
    }
    for row in selected_rows.to_dict('records')
}

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'wf2020_riskcap_full_bc_replay_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
    'closest_historical_success_case': 'bull_reentry_three_fold_checkpoint_metric_replay_v1 passed wf_2022 and wf_2023_2024 but failed wf_2020 return.',
    'closest_historical_failure_case': 'wf2020_riskcap_full_bc_replay_v1 passed 3/3 MaxDD gates but failed return/relative gates through underparticipation.',
    'why_not_duplicate': 'This card tests hybrid target contracts plus class-balanced BC weighting, not the full risk-cap replacement already replayed.',
    'changed_layer': ['model_selection', 'training_target', 'bc_warm_start', 'sample_weighting', 'official_replay'],
    'candidate_variants': candidate_variants,
    'weight_configs': {name: asdict(config) for name, config in WEIGHT_CONFIGS.items()},
    'selected_candidate': selected_candidate,
    'selected_variant': selected_variant,
    'selected_target_contract': selected_contract_path,
    'target_contract_gate': bool(full_gate),
    'bc_readiness_gate': bool(full_gate),
    'bc_screen_allowed_next': bool(full_gate),
    'target_max_dd_pct': TARGET_MAX_DD,
    'target_relative_return_by_fold_pp': gate_contract.get('target_relative_return_by_fold_pp', {}),
    'required_wf_windows': required_folds,
    'folds_evaluated': selected_rows['fold'].astype(str).tolist() if not selected_rows.empty else [],
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'fold_gate_pass_count': int(selected_rows['fold_gate'].sum()) if not selected_rows.empty else 0,
    'max_dd_pass_count': max_dd_pass_count,
    'relative_pass_count': relative_pass_count,
    'return_floor_pass_count': return_floor_pass_count,
    'selected_by_fold': selected_by_fold,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': bool(full_gate),
    'next_training_allowed_after_replay': bool(full_gate),
    'promotion_blocker': 'Diagnostic target/BC replay only. No retained actor bundle, fixed-split/YTD replay, or promotion review is produced in this card.',
    'next_mechanism': next_mechanism,
    'source_decisions': {
        'wf2020_risk_cap_reference_target_screen_v1': source_summary.get('decision'),
        'wf2020_riskcap_full_bc_replay_v1': direct_replay_summary.get('decision'),
        'bull_reentry_three_fold_checkpoint_metric_replay_v1': 'three_fold_checkpoint_metric_replay_partial_pass_no_promotion',
    },
    'sidecars': [
        'candidate_contract_manifest.csv',
        'candidate_target_distribution.csv',
        'candidate_weight_config_manifest.csv',
        'coverage.csv',
        'bc_fit_metrics.csv',
        'bc_slice_metrics.csv',
        'bc_action_distribution.csv',
        'fold_candidate_replay_metrics.csv',
        'candidate_scorecard.csv',
        'fold_replay_metrics.csv',
        'drawdown_interval_diagnostic.csv',
        'daily_path_manifest.csv',
        '<candidate>_<fold>_official_daily_path.csv',
        'selected_hybrid_target_contract.csv',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}

summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'created_at': selection['created_at'],
    'canonical_baseline': selection['canonical_baseline'],
    'direct_comparison_baseline': selection['direct_comparison_baseline'],
    'reference_baseline': selection['reference_baseline'],
    'role': selection['role'],
    'changed_layer': selection['changed_layer'],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': bool(full_gate),
    'next_training_allowed_after_replay': bool(full_gate),
    'target_contract_gate': bool(full_gate),
    'bc_readiness_gate': bool(full_gate),
    'bc_screen_allowed_next': bool(full_gate),
    'next_mechanism': next_mechanism,
    'selected_candidate': selected_candidate,
    'selected_variant': selected_variant,
    'selected_target_contract': selected_contract_path,
    'target_max_dd_pct': TARGET_MAX_DD,
    'target_relative_return_by_fold_pp': selection['target_relative_return_by_fold_pp'],
    'required_best_return_pct': gate_contract.get('required_ytd_return_pct'),
    'required_soxlbacktest_return_pct': gate_contract.get('required_soxlbacktest_reference_return_pct'),
    'wf2020_reference_floor_return_pct': gate_contract.get('wf2020_reference_floor_return_pct'),
    'wf2020_reference_floor_max_dd_pct': gate_contract.get('wf2020_reference_floor_max_dd_pct'),
    'required_wf_windows': required_folds,
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'selected_by_fold': selected_by_fold,
    'fold_gate_pass_count': selection['fold_gate_pass_count'],
    'max_dd_pass_count': max_dd_pass_count,
    'relative_pass_count': relative_pass_count,
    'return_floor_pass_count': return_floor_pass_count,
    'source_summary': {
        'riskcap_screen_decision': source_summary.get('decision'),
        'full_replay_decision': direct_replay_summary.get('decision'),
        'full_replay_fold_gate_pass_count': direct_replay_summary.get('fold_gate_pass_count'),
        'full_replay_max_dd_pass_count': direct_replay_summary.get('max_dd_pass_count'),
        'full_replay_relative_pass_count': direct_replay_summary.get('relative_pass_count'),
    },
    'source_decisions': selection['source_decisions'],
    'candidate_scorecard': candidate_score.to_dict('records') if not candidate_score.empty else [],
    'fold_results': selected_rows.to_dict('records'),
    'bc_train_config': asdict(BC_TRAIN_CONFIG),
    'sidecars': selection['sidecars'],
    'conclusion': [],
}

if selected_rows.empty:
    summary['conclusion'].append('No candidates evaluated; do not continue this branch.')
else:
    summary['conclusion'].append(f"Selected global candidate `{selected_candidate}` from {len(candidate_score)} candidate groups.")
    for row in selected_rows.to_dict('records'):
        summary['conclusion'].append(
            f"{row['fold']}: return={row['return']:.2f}%, relative={row['relative_return']:.2f}pp "
            f"(target {row['target_relative_return']:.2f}pp), MaxDD={row['max_dd']:.2f}% "
            f"(<{TARGET_MAX_DD:.2f}% gate={bool(row['max_dd_gate'])}), avg_pos={row['avg_pos']:.3f}, "
            f"fold_gate={bool(row['fold_gate'])}."
        )
    if full_gate:
        summary['conclusion'].append('All required folds passed the diagnostic target+BC replay gates, but no runtime promotion is allowed until fixed-split/YTD actor retention review is run.')
    else:
        summary['conclusion'].append('No runtime promotion: the selected hybrid still fails at least one required fold gate.')

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
conclusion_lines = [
    f'# {EXPERIMENT}',
    '',
    f'Decision: `{decision}`.',
    '',
    f'Selected candidate: `{selected_candidate or "none"}`.',
    '',
    f'Fold gates passed: `{selection["fold_gate_pass_count"]}/{len(required_folds)}`. MaxDD gates passed: `{max_dd_pass_count}/{len(required_folds)}`. Relative gates passed: `{relative_pass_count}/{len(required_folds)}`.',
    '',
    *[f'- {line}' for line in summary['conclusion']],
    '',
    f'Next action: `{next_mechanism}`.',
    '',
    'Promotion note: `promotion_allowed=false`, `actor_retention_allowed=false`; this card is a diagnostic replay, not a retained model.',
]
(CARD / 'conclusion.md').write_text('\n'.join(conclusion_lines) + '\n', encoding='utf-8')
summary


{'experiment': 'return_preserving_hybrid_target_bc_replay_v1',
 'decision': 'return_preserving_hybrid_bc_replay_no_promotion',
 'created_at': '2026-06-14T01:36:20.435158+00:00',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'wf2020_riskcap_full_bc_replay_v1',
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
 'role': 'diagnose',
 'changed_layer': ['model_selection',
  'training_target',
  'bc_warm_start',
  'sample_weighting',
  'official_replay'],
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'next_training_allowed_after_replay': False,
 'target_contract_gate': False,
 'bc_readiness_gate': False,
 'bc_screen_allowed_next': False,
 'next_mechanism': 'inspect_selected_hybrid_failure_and_repair_participation_weight_or_target_before_retention_training',
 'selected_candidate': 'hybrid_blend_nonrisk_50__riskcap_default_weight',
 'selected_variant': 'hy

## Conclusion

This cell is populated by the executed notebook. The sidecar `summary.json`, `selection.json`, and `conclusion.md` are written by the final code cell.
